<a href="https://colab.research.google.com/github/viejomatius/pedidos_ia_tesis_mia/blob/main/main_procesamiento_pedidos_ia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Tesis MIA Aplicada - Modulo Inteligente de Procesamiento de Pedidos**
###**Estado del Desarrollo:**

Prueba de Concepto (Proof of Concept - PoC) / Versión Inicial.

<details>
<summary><b>Clic aqui para ver la información del proyecto de tesis</b></summary>
<br>

* **Proyecto de Tesis:** Desarrollo de modulo inteligente de procesamiento de pedidos mediante modelos de lenguaje y visión artificial

* **Autores:** Cordova Mateo, Portero Juan

* **Programa:** Maestría en Inteligencia Artificial Aplicada - Universidad de las Americas

* **Año:** 2026

* **Estado del Desarrollo:** Prueba de Concepto (Proof of Concept - PoC) / Versión Inicial.

### **Resumen Ejecutivo**
Esta prueba de concepto (PoC) automatiza el procesamiento de órdenes de compra informales utilizando visión artificial (OCR) e IA Generativa (LLM + RAG). Su objetivo es eliminar la entrada manual de datos para reducir la carga administrativa, minimizar errores y acelerar los tiempos de respuesta (Lead Time) del equipo comercial.

<details>
<summary><b>Clic aquí para ver: Contexto del problema, Objetivo del Notebook, Alcance Actual, Roadmap, Arquitectura y Metricas</b></summary>

<br>

### **Contexto del Problema**
La empresa Ideal Alambrec recibe un alto volumen de pedidos a través de canales digitales informales (fotografías, WhatsApp, notas a mano).Existe una "brecha semántica" entre la descripción coloquial del cliente (ej. "alambre delgadito") y la nomenclatura técnica del catálogo (SKU). Actualmente, el equipo comercial actúa como un "middleware humano" para interpretar estos pedidos, un proceso manual que es altamente propenso a errores, genera retrasos operativos y aumenta las tasas de devoluciones logísticas.

### **Objetivo del Notebook**
El objetivo de este documento es validar la viabilidad técnica del módulo central de procesamiento. Se implementa y evalúa un pipeline de IA que extrae texto de imágenes, recupera contexto relevante de un catálogo vectorizado, y utiliza un LLM mediante *Few-Shot Prompting* para normalizar los datos en un formato JSON estructurado. Adicionalmente, se integra una lógica de enrutamiento (*Human-in-the-Loop*) para clasificar los pedidos según el nivel de confianza del modelo.

###**Alcance Actual (Proof of Concept)**
**Implementado:**
* Pipeline de preprocesamiento de imágenes con OpenCV y extracción de texto con Tesseract.
* Base de datos vectorial simulada en memoria (FAISS) con metadatos enriquecidos (sinónimos).
* Lógica de recuperación y generación aumentada (RAG) utilizando `gpt-4o`.
* Enrutamiento condicional de la confianza (*Human-in-the-Loop*).
* Evaluación de métricas de rendimiento académico (F1-Score, Exactitud).

**Experimental / A mejorar en la tesis:**
* El catálogo es un subconjunto simulado (Hardcoded); deberá conectarse a la base de datos SQL o ERP real.
* La inyección de la API Key se realiza de forma directa para fines de esta prueba y debe migrar a variables de entorno seguras.
* El OCR de Tesseract se evaluará frente a soluciones cloud como Azure Vision en fases posteriores.

### **Roadmap de Evolución del Código (12 Semanas)**
* **Semanas 1-3 (Datos y Taxonomía):** (explicar que implica la taxonomia) Sustituir el catálogo simulado por el corpus histórico real anonimizado. Refinar la lista de sinónimos y variabilidad lingüística.
* **Semanas 4-6 (Arquitectura RAG):** Optimizar la técnica de *chunking* y *embeddings*. Migrar FAISS a una base de datos vectorial persistente (ej. ChromaDB/Pinecone).
* **Semanas 7-9 (Visión Artificial y OCR):** Implementar corrección de perspectiva en OpenCV y comparar el rendimiento de Tesseract vs. Azure Document Intelligence.
* **Semanas 10-12 (Despliegue y HITL):** Empaquetar el pipeline en una API REST (FastAPI) y construir el frontend para la interfaz de validación humana (*Human-in-the-loop*).

### **Arquitectura General de la Solución**
El flujo del sistema, basado en una arquitectura multimodal, sigue los siguientes pasos secuenciales:
1.  **Entrada de Datos:** Recepción de la orden de compra (texto plano o imagen).
2.  **Procesamiento de Visión:** Binarización adaptativa mediante OpenCV y reconocimiento óptico de caracteres (OCR).
3.  **Recuperación Semántica (RAG):** El texto extraído se vectoriza y busca en FAISS las 3 mayores coincidencias del catálogo.
4.  **Inferencia LLM:** El modelo generativo recibe el contexto, evalúa la intención mediante ejemplos (*Few-Shot*), y devuelve un objeto JSON.
5.  **Reglas de Negocio:** El sistema evalúa la llave `"confianza"`. Si es alta, se enruta al ERP; si es baja, se marca para revisión manual.

### **Métricas de Éxito del Proyecto**
Para validar el sistema técnica y comercialmente, se utilizarán las siguientes métricas:
* **Exactitud (Accuracy):** Porcentaje de pedidos mapeados al SKU correcto sobre el total.
* **F1-Score Ponderado:** Métrica robusta ante el desbalance de clases (productos muy vendidos vs. poco vendidos). Calculada mediante la fórmula $F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$.
* **Tiempo de Inferencia:** Segundos promedio requeridos por el sistema para resolver una orden.
* **Reducción del Lead Time (Métrica de Negocio):** Tiempo ahorrado en digitación manual por el equipo comercial.

---
### **Glosario Técnico**
* **LLM (Large Language Model):** Modelo de IA entrenado para comprender y generar lenguaje natural (ej. GPT-4o).
* **RAG (Retrieval-Augmented Generation):** Arquitectura que enriquece un LLM proporcionándole datos externos (como un catálogo) antes de generar una respuesta.
* **FAISS:** Librería de Facebook AI para la búsqueda de similitud eficiente y agrupamiento de vectores densos.
* **Few-Shot Prompting:** Técnica donde se provee al modelo de lenguaje con una pequeña cantidad de ejemplos de entrenamiento en su instrucción para mejorar la precisión de la inferencia.
* **Human-in-the-Loop (HITL):** Patrón de diseño de IA que requiere interacción humana para resolver ambigüedades operativas o decisiones de baja confianza.

### **Sección de Código**

In [ ]:
# @title **Generacion de Catálogo Sintetico (Simulacion de data de ERP con inventario)**
# ==============================================================================
# MÓDULO 1: GENERACIÓN DE CATÁLOGO SINTÉTICO (SIMULACIÓN ERP CON INVENTARIO)
# ==============================================================================
import pandas as pd
import numpy as np

def generar_catalogo_b2b_con_stock(num_registros: int = 150, nombre_archivo: str = "catalogo_erp_ideal.csv"):
    """
    Genera un DataFrame con productos B2B, incluyendo simulación de inventario (Stock).
    Diseñado sin contaminación conversacional para pruebas de RAG Híbrido.
    """
    np.random.seed(42) # Semilla fija para reproducibilidad académica

    categorias = {
        'Alambre de Púas': {'prefijo': 'ALM', 'unidades': 'Rollo', 'jergas': ['pua', 'alambre de puas', 'pua perimetral', 'alambre para cercar', 'alambre con espinas', 'alambre espinoso']},
        'Malla Electrosoldada': {'prefijo': 'MLE', 'unidades': 'Plancha', 'jergas': ['malla de construccion', 'malla soldada', 'parrilla', 'malla para hormigon', 'rejilla de metal', 'malla de acero']},
        'Postes de Concreto': {'prefijo': 'PST', 'unidades': 'Unidad', 'jergas': ['poste de cemento', 'pilar', 'poste para cerca', 'poste para alambrado', 'viga de hormigon']},
        'Grapas Galvanizadas': {'prefijo': 'GRP', 'unidades': 'Caja', 'jergas': ['grapas', 'clavos en U', 'grampas', 'sujetadores de alambre', 'grapas para cerca']},
        'Clavos de Acero': {'prefijo': 'CLV', 'unidades': 'Caja', 'jergas': ['clavos', 'clavo corrugado', 'clavo para madera', 'clavos fuertes', 'puntillas de acero']}
    }

    calibres_o_tamanos = ['Fino', 'Estándar', 'Grueso', 'Calibre 12', 'Calibre 14', 'Alta Resistencia']
    datos = []

    for i in range(1, num_registros + 1):
        cat_nombre = np.random.choice(list(categorias.keys()))
        cat_info = categorias[cat_nombre]

        sku = f"{cat_info['prefijo']}-{1000 + i}"
        tamano = np.random.choice(calibres_o_tamanos)
        nombre_tecnico = f"{cat_nombre} {tamano} Tipo {np.random.randint(1, 5)}"
        jerga_base = np.random.choice(cat_info['jergas'])

        # ⚠️ CORRECCIÓN ARQUITECTÓNICA: Solo variaciones semánticas del producto. Cero intenciones/verbos.
        descripcion_jerga = np.random.choice([
            f"{jerga_base} {tamano.lower()}",
            f"{jerga_base} del tipo {tamano.lower()}",
            f"{jerga_base} de {tamano.lower()} (el normal)",
            f"{jerga_base} ({tamano.lower()})"
        ])

        precio = round(np.random.uniform(15.0, 250.0), 2)

        # Lógica de Negocio: Simulación de inventario (Quiebres de stock controlados)
        stock = np.random.choice(
            [0, np.random.randint(5, 50), np.random.randint(100, 500)],
            p=[0.1, 0.4, 0.5]
        )

        datos.append({
            "SKU": sku,
            "Nombre_Tecnico": nombre_tecnico,
            "Categoria": cat_nombre,
            "Descripcion_Jerga": descripcion_jerga,
            "Unidad": cat_info['unidades'],
            "Precio_Referencia": precio,
            "Stock_Disponible": stock
        })

    df_catalogo = pd.DataFrame(datos)
    df_catalogo.to_csv(nombre_archivo, index=False, encoding='utf-8')
    print(f"✅ ¡Éxito! Catálogo sintético purificado generado con {num_registros} productos.")
    print(f"📊 Estadísticas de Inventario: {(df_catalogo['Stock_Disponible'] == 0).sum()} productos están agotados (Stock 0).")

    return df_catalogo, nombre_archivo

# Ejecutar y mostrar
df_catalogo, nombre_archivo = generar_catalogo_b2b_con_stock(150)
display(df_catalogo.head(5))

✅ ¡Éxito! Catálogo sintético purificado generado con 150 productos.
📊 Estadísticas de Inventario: 15 productos están agotados (Stock 0).


,SKU,Nombre_Tecnico,Categoria,Descripcion_Jerga,Unidad,Precio_Referencia,Stock_Disponible
0,GRP-1001,Grapas Galvanizadas Calibre 14 Tipo 3,Grapas Galvanizadas,grampas (calibre 14),Caja,155.68,43
1,PST-1002,Postes de Concreto Grueso Tipo 4,Postes de Concreto,viga de hormigon (grueso),Unidad,48.57,0
2,GRP-1003,Grapas Galvanizadas Alta Resistencia Tipo 2,Grapas Galvanizadas,clavos en U (alta resistencia),Caja,248.17,413
3,GRP-1004,Grapas Galvanizadas Fino Tipo 1,Grapas Galvanizadas,grampas de fino (el normal),Caja,108.97,32
4,PST-1005,Postes de Concreto Calibre 12 Tipo 3,Postes de Concreto,poste para alambrado (calibre 12),Unidad,124.69,406


In [ ]:
# @title **MÓDULO 1: CONFIGURACIÓN, SEGURIDAD Y OBSERVABILIDAD (NIVEL PRODUCCIÓN)**
# ==============================================================================
# MÓDULO 1: CONFIGURACIÓN, SEGURIDAD Y OBSERVABILIDAD (NIVEL PRODUCCIÓN)
# ==============================================================================
import os
import logging
import subprocess
from google.colab import userdata

# 1. Configuración de Logs Estructurados
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("IA_Pedidos_IdealAlambrec")

# 2. Instalación de Dependencias OS (OCR)
logger.info("Instalando motor OCR a nivel de sistema...")
subprocess.run("sudo apt-get install tesseract-ocr tesseract-ocr-spa -y", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Instalación de Ecosistema Python (Incluyendo rank_bm25 para Búsqueda Híbrida)
logger.info("Instalando librerías de IA, Visión y MLOps...")
subprocess.run("pip install langchain langchain-openai langchain-community faiss-cpu tiktoken gradio pytesseract opencv-python-headless pandas scikit-learn rank_bm25 -q", shell=True)

# 4. Gestión Segura de Credenciales
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    logger.info("✅ API Key cargada de forma segura desde Colab Secrets.")
except Exception as e:
    logger.error("❌ No se encontró OPENAI_API_KEY. Configúrala en la barra lateral izquierda (Secretos).")

In [ ]:
# @title **MÓDULO 2 V4: INGESTA ERP Y BÚSQUEDA HÍBRIDA NATIVA (SIN DEPENDENCIAS ROTAS)**
# ==============================================================================
# MÓDULO 2 V4: INGESTA ERP Y BÚSQUEDA HÍBRIDA NATIVA (SIN DEPENDENCIAS ROTAS)
# ==============================================================================
import pandas as pd
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# ⚠️ SOLUCIÓN DE ARQUITECTURA: Clase personalizada para evitar el Dependency Hell de LangChain
class BuscadorHibridoPersonalizado:
    def __init__(self, faiss_retriever, bm25_retriever):
        self.faiss_retriever = faiss_retriever
        self.bm25_retriever = bm25_retriever

    def invoke(self, query: str):
        """
        Ejecuta ambas búsquedas en paralelo y combina los resultados,
        eliminando documentos duplicados basados en su SKU.
        """
        docs_semanticos = self.faiss_retriever.invoke(query)
        docs_lexicos = self.bm25_retriever.invoke(query)

        documentos_finales = []
        skus_vistos = set()

        # Combinar resultados priorizando la búsqueda semántica, luego la léxica
        for doc in docs_semanticos + docs_lexicos:
            sku = doc.metadata.get("sku")
            if sku not in skus_vistos:
                skus_vistos.add(sku)
                documentos_finales.append(doc)

        return documentos_finales

def inicializar_motor_busqueda_hibrido(ruta_csv: str):
    logger.info(f"Iniciando ingesta de datos desde: {ruta_csv}")

    try:
        df_erp = pd.read_csv(ruta_csv)
    except FileNotFoundError:
        logger.error(f"El archivo {ruta_csv} no existe. Asegúrate de ejecutar el Módulo 1 generador.")
        raise

    # 1. Mapeo 1:1 - Preservación semántica tabular
    documentos_enriquecidos = []
    for _, row in df_erp.iterrows():
        texto_vector = (
            f"SKU: {row['SKU']} | Producto Técnico: {row['Nombre_Tecnico']} | "
            f"Categoría: {row['Categoria']} | Jerga y Sinónimos: {row['Descripcion_Jerga']} | "
            f"Unidad de Venta: {row['Unidad']}"
        )
        doc = Document(page_content=texto_vector, metadata={"sku": row['SKU']})
        documentos_enriquecidos.append(doc)

    # 2. Retenedor Semántico (FAISS - Dense)
    logger.info("Construyendo índice FAISS (Semántico)...")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = FAISS.from_documents(documentos_enriquecidos, embeddings)
    faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    # 3. Retenedor Léxico (BM25 - Sparse)
    logger.info("Construyendo índice BM25 (Coincidencia Exacta)...")
    bm25_retriever = BM25Retriever.from_documents(documentos_enriquecidos)
    bm25_retriever.k = 4

    # 4. Instanciar nuestro Ensamble Personalizado
    retriever_hibrido = BuscadorHibridoPersonalizado(faiss_retriever, bm25_retriever)

    logger.info("✅ Motor Híbrido Nativo inicializado con éxito.")
    return retriever_hibrido, df_erp

# Ejecutar Inicialización
retriever_hibrido, df_erp = inicializar_motor_busqueda_hibrido("catalogo_erp_ideal.csv")

In [ ]:
# @title **MÓDULO 3 V5: PIPELINE RAG CON FEW-SHOT PROMPTING**
# ==============================================================================
# MÓDULO 3 V5: PIPELINE RAG CON FEW-SHOT PROMPTING
# ==============================================================================
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_community.callbacks import get_openai_callback

def procesar_pedido_produccion(texto_cliente: str, retriever, df_inventario: pd.DataFrame) -> dict:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # 1. Recuperación Híbrida
    docs = retriever.invoke(texto_cliente)
    contexto = "\n".join([d.page_content for d in docs])

    # 2. Prompt Estricto con Few-Shot Prompting
    template = """
    Eres un analista B2B senior. Mapea el pedido del cliente usando ÚNICAMENTE este catálogo oficial:
    {contexto}

    PEDIDO DEL CLIENTE: "{pedido}"

    EJEMPLOS DE RAZONAMIENTO (FEW-SHOT):
    - Input: "dame 2 rollos de pua y pintura"
      Razonamiento: "Pua coincide con el SKU de Alambre de Púas. 'Pintura' no existe en el catálogo, requiere revisión."
      Salida JSON: [{{"sku": "ALM-XXXX", "cantidad_solicitada": 2, "confianza_ia": "Alta", "razonamiento": "..."}}, {{"sku": "REVISION_MANUAL", "cantidad_solicitada": 1, "confianza_ia": "Baja", "razonamiento": "Pintura fuera de catálogo"}}]

    - Input: "necesito postes"
      Razonamiento: "El cliente pide postes pero no especifica el tamaño o calibre. Hay varios en el catálogo. Es ambiguo."
      Salida JSON: [{{"sku": "REVISION_MANUAL", "cantidad_solicitada": 1, "confianza_ia": "Baja", "razonamiento": "Ambigüedad en tipo de poste"}}]

    INSTRUCCIONES CRÍTICAS:
    1. Si es ambiguo, vago, o no está en el catálogo recuperado, asigna SKU: "REVISION_MANUAL" y confianza "Baja".
    2. Si hay coincidencia exacta (incluso usando la jerga del catálogo), asigna el SKU recuperado y confianza "Alta".
    3. Devuelve EXCLUSIVAMENTE un JSON válido con la estructura: {{"pedidos": [...]}}
    """
    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm | JsonOutputParser()

    with get_openai_callback() as cb:
        try:
            resultado_ia = chain.invoke({"contexto": contexto, "pedido": texto_cliente})
            costo_usd = cb.total_cost
            tokens_totales = cb.total_tokens
        except Exception as e:
            return {"pedidos": [], "metadatos_operativos": {"requiere_hitl": True, "error": str(e), "costo_usd": 0.0, "tokens_consumidos": 0}}

    # Lógica Transaccional (Cruce ERP y Filtro de Seguridad)
    pedidos_procesados = []
    requiere_hitl = False

    for item in resultado_ia.get("pedidos", []):
        if item.get("confianza_ia") in ["Baja", "Media"]:
            item["sku"] = "REVISION_MANUAL"
            item["confianza_ia"] = "Baja"

        sku = item.get("sku")
        cantidad = item.get("cantidad_solicitada", 0)

        if sku == "REVISION_MANUAL":
            item["estado_sistema"] = "⚠️ REVISIÓN HUMANA (Ambigüedad)"
            requiere_hitl = True
        else:
            producto_db = df_inventario[df_inventario['SKU'] == sku]
            if not producto_db.empty:
                stock_real = producto_db.iloc[0]['Stock_Disponible']
                if stock_real >= cantidad:
                    item["estado_sistema"] = "✅ APROBADO (Directo a ERP)"
                else:
                    item["estado_sistema"] = f"❌ QUIEBRE DE STOCK (Disp: {stock_real})"
                    requiere_hitl = True
            else:
                item["estado_sistema"] = "❌ ERROR (SKU Inexistente)"
                item["sku"] = "REVISION_MANUAL"
                requiere_hitl = True

        pedidos_procesados.append(item)

    return {
        "pedidos": pedidos_procesados,
        "metadatos_operativos": {
            "requiere_hitl": requiere_hitl, "costo_usd": costo_usd, "tokens_consumidos": tokens_totales
        }
    }

In [ ]:
# @title **MÓDULO 4 V5: BOOTSTRAPPING DINÁMICO Y EVALUACIÓN MULTI-ETIQUETA**
# ==============================================================================
# MÓDULO 4 V5: BOOTSTRAPPING DINÁMICO Y EVALUACIÓN MULTI-ETIQUETA
# ==============================================================================
import time
import pandas as pd
import numpy as np
import random

def generar_dataset_dinamico(df_erp):
    """Genera casos de prueba dinámicos asegurando que los SKUs existan realmente."""
    casos = []

    # Caso 1: Multi-producto claro (Happy Path)
    p1 = df_erp[df_erp['Categoria'] == 'Alambre de Púas'].sample(1).iloc[0]
    p2 = df_erp[df_erp['Categoria'] == 'Grapas Galvanizadas'].sample(1).iloc[0]
    casos.append({
        "texto": f"Cotízame 5 rollos de {p1['Descripcion_Jerga']} y 2 cajas de {p2['Descripcion_Jerga']}.",
        "skus_reales": [p1['SKU'], p2['SKU']]
    })

    # Caso 2: Un producto con jerga compleja
    p3 = df_erp[df_erp['Categoria'] == 'Malla Electrosoldada'].sample(1).iloc[0]
    casos.append({
        "texto": f"Mándame 10 planchas de {p3['Descripcion_Jerga']} urgente.",
        "skus_reales": [p3['SKU']]
    })

    # Caso 3: Mezcla de producto real + producto fuera de dominio
    p4 = df_erp[df_erp['Categoria'] == 'Clavos de Acero'].sample(1).iloc[0]
    casos.append({
        "texto": f"Quiero 100 cajas de {p4['Descripcion_Jerga']} y también 5 cascos de seguridad.",
        "skus_reales": [p4['SKU'], "REVISION_MANUAL"]
    })

    # Caso 4 y 5: Ambigüedad total (Forzando el HITL)
    casos.append({"texto": "necesito postes para la cerca, unos 20", "skus_reales": ["REVISION_MANUAL"]})
    casos.append({"texto": "tienes tubos de PVC y cemento?", "skus_reales": ["REVISION_MANUAL"]})

    return casos

def evaluar_pipeline_multietiqueta(dataset_pruebas: list, retriever, df_erp: pd.DataFrame):
    logger.info(f"Ejecutando evaluación sobre {len(dataset_pruebas)} canastas de prueba generadas dinámicamente...")

    metricas_canasta, tiempos_ejecucion = [], []
    costo_acumulado = 0.0

    for caso in dataset_pruebas:
        inicio = time.time()
        respuesta = procesar_pedido_produccion(caso['texto'], retriever, df_erp)
        tiempos_ejecucion.append(time.time() - inicio)
        costo_acumulado += respuesta.get("metadatos_operativos", {}).get("costo_usd", 0.0)

        set_true = set(caso['skus_reales'])
        set_pred = set([p.get("sku") for p in respuesta.get("pedidos", []) if p.get("sku")])

        # Corrección lógica para evaluación justa de HITL
        if "REVISION_MANUAL" in set_true:
            if any(item.get("sku") == "REVISION_MANUAL" for item in respuesta.get("pedidos", [])):
                set_true.add("HITL_CORRECTO")
                set_pred.add("HITL_CORRECTO")
            set_true.discard("REVISION_MANUAL")
            set_pred.discard("REVISION_MANUAL")

        interseccion = len(set_true.intersection(set_pred))
        precision = interseccion / len(set_pred) if len(set_pred) > 0 else 0.0
        recall = interseccion / len(set_true) if len(set_true) > 0 else 0.0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        metricas_canasta.append({"precision": precision, "recall": recall, "f1": f1_score})

    df_metricas = pd.DataFrame(metricas_canasta)

    print("\n" + "="*16 + " RESULTADOS TÉCNICOS " + "="*16)
    print(f"Precisión Global:   {df_metricas['precision'].mean():.2%}")
    print(f"Recall Global:      {df_metricas['recall'].mean():.2%}")
    print(f"F1-Score Global:    {df_metricas['f1'].mean():.4f}")
    print("="*16 + " METRICAS DE NEGOCIO " + "="*16)
    print(f"Latencia promedio:  {np.mean(tiempos_ejecucion):.2f} s por pedido")
    print(f"Costo API total:    ${costo_acumulado:.5f} USD")
    print("="*53 + "\n")

# Ejecutar el Test Dinámico
dataset_dinamico = generar_dataset_dinamico(df_erp)
evaluar_pipeline_multietiqueta(dataset_dinamico, retriever_hibrido, df_erp)


================ RESULTADOS TÉCNICOS ================
Precisión Global:   80.00%
Recall Global:      80.00%
F1-Score Global:    0.8000
================ METRICAS DE NEGOCIO ================
Latencia promedio:  3.39 s por pedido
Costo API total:    $0.00081 USD



In [ ]:
# @title **MÓDULO 5 Y 6: VISIÓN ARTIFICIAL (OPENCV) Y DATA FLYWHEEL (HITL)**
# ==============================================================================
# MÓDULO 5 Y 6: VISIÓN ARTIFICIAL (OPENCV) Y DATA FLYWHEEL (JSONL)
# ==============================================================================
import cv2
import numpy as np
import pytesseract
import json
from datetime import datetime

# --- Módulo 5: Visión Artificial ---
def extraer_texto_desde_imagen(imagen_numpy) -> str:
    if imagen_numpy is None: return ""
    try:
        img_bgr = cv2.cvtColor(imagen_numpy, cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
        return pytesseract.image_to_string(thresh, lang='spa+eng').strip()
    except Exception as e:
        logger.error(f"Error OpenCV/OCR: {e}")
        return f"ERROR_OCR"

# --- Módulo 6: Base de Datos de Retroalimentación (HITL) ---
ARCHIVO_DATASET_HITL = "dataset_finetuning_hitl.jsonl"

def guardar_feedback_humano(texto_cliente: str, json_ia: str, json_corregido: str) -> str:
    if not texto_cliente or not json_corregido:
        return "⚠️ Faltan datos para guardar."
    registro = {
        "timestamp": datetime.now().isoformat(),
        "input_crudo": texto_cliente,
        "prediccion_ia": json_ia,
        "ground_truth_humano": json_corregido
    }
    try:
        with open(ARCHIVO_DATASET_HITL, "a", encoding="utf-8") as f:
            f.write(json.dumps(registro, ensure_ascii=False) + "\n")
        return "✅ Feedback guardado para Fine-Tuning."
    except Exception as e:
        return f"❌ Error guardando: {e}"

In [ ]:
# @title **MÓDULO 7: INTERFAZ GERENCIAL Y PANEL HITL (GRADIO)**
# ==============================================================================
# MÓDULO 7: INTERFAZ GERENCIAL Y PANEL HITL (FRONTEND)
# ==============================================================================
import gradio as gr
import json # Ensure json is imported for dumps

# --- Controladores ---
def ejecutar_pipeline_ui(texto):
    resultado = procesar_pedido_produccion(texto, retriever_hibrido, df_erp)
    json_formateado = json.dumps(resultado["pedidos"], indent=2, ensure_ascii=False)
    metadatos = resultado["metadatos_operativos"]
    estado_ui = "⚠️ REQUIERE ACCIÓN HUMANA" if metadatos["requiere_hitl"] else "✅ APROBADO (DIRECTO ERP)"

    # New logic to extract and format the first item
    first_item_summary = "No se procesaron artículos." # Default message
    if resultado["pedidos"]:
        first_ped = resultado["pedidos"][0]
        sku = first_ped.get("sku", "N/A")
        cantidad = first_ped.get("cantidad_solicitada", "N/A")
        confianza = first_ped.get("confianza_ia", "N/A")
        estado_sys = first_ped.get("estado_sistema", "N/A")

        first_item_summary = (
            f"Primer Artículo Procesado:\n"
            f"  SKU: {sku}\n"
            f"  Cantidad Solicitada: {cantidad}\n"
            f"  Confianza IA: {confianza}\n"
            f"  Estado del Sistema: {estado_sys}"
        )

    # Return the new summary along with existing outputs
    return texto, json_formateado, estado_ui, f"${metadatos['costo_usd']:.5f}", str(metadatos['tokens_consumidos']), first_item_summary

def procesar_UI_texto(txt):
    if txt.strip():
        # Unpack all returned values, including the new first_item_summary
        txt_out, json_out, estado_out, costo_out, tokens_out, first_item_out = ejecutar_pipeline_ui(txt)
        return txt_out, json_out, estado_out, costo_out, tokens_out, first_item_out
    else:
        # Ensure all outputs are handled, including the new one
        return ("N/A", "Ingrese pedido.", "N/A", "N/A", "N/A", "N/A")

def procesar_UI_imagen(img):
    txt_ocr = extraer_texto_desde_imagen(img)
    if "ERROR" in txt_ocr or len(txt_ocr) < 3:
        # Ensure all outputs are handled, including the new one
        return txt_ocr, "Fallo OCR.", "🔴 ERROR", "N/A", "N/A", "N/A"

    # Unpack all returned values, including the new first_item_summary
    txt_out, json_out, estado_out, costo_out, tokens_out, first_item_out = ejecutar_pipeline_ui(txt_ocr)
    return txt_out, json_out, estado_out, costo_out, tokens_out, first_item_out

# --- Construcción UI ---
with gr.Blocks(theme=gr.themes.Soft()) as app_produccion:
    gr.Markdown("# 🏭 Inteligencia Artificial B2B - Ideal Alambrec")

    with gr.Row():
        with gr.Column():
            with gr.Tabs():
                with gr.TabItem("📝 Texto"):
                    in_txt = gr.Textbox(lines=4, label="Mensaje del Cliente")
                    btn_txt = gr.Button("Procesar", variant="primary")
                with gr.TabItem("📸 Imagen"):
                    in_img = gr.Image(label="Subir Nota/Foto")
                    out_ocr = gr.Textbox(label="Texto OCR Extraído", interactive=False)
                    btn_img = gr.Button("OCR + Procesar", variant="primary")

        with gr.Column():
            out_estado = gr.Textbox(label="Decisión Transaccional")
            out_json = gr.Code(label="Payload ERP (JSON)", language="json")
            # New Textbox for displaying the first item's summary
            out_first_item = gr.Textbox(label="Detalle Primer Artículo", interactive=False, lines=6)
            with gr.Row():
                out_costo = gr.Textbox(label="Costo USD")
                out_tokens = gr.Textbox(label="Tokens")

    gr.Markdown("---")
    gr.Markdown("### 🧑‍💻 Auditoría Humana (Human-in-the-Loop)")

    with gr.Row():
        in_hitl = gr.Code(label="Corregir JSON (Entrenamiento)", language="json", interactive=True)
    with gr.Row():
        btn_hitl = gr.Button("💾 Guardar en Dataset", variant="secondary")
        out_hitl = gr.Textbox(label="Estado")

    # --- Eventos ---
    # Update outputs for btn_txt.click to include out_first_item
    btn_txt.click(fn=procesar_UI_texto, inputs=in_txt, outputs=[in_txt, out_json, out_estado, out_costo, out_tokens, out_first_item]).then(fn=lambda x: x, inputs=out_json, outputs=in_hitl)
    # Update outputs for btn_img.click to include out_first_item
    btn_img.click(fn=procesar_UI_imagen, inputs=in_img, outputs=[out_ocr, out_json, out_estado, out_costo, out_tokens, out_first_item]).then(fn=lambda x: x, inputs=out_json, outputs=in_hitl)

    def wrapper_guardar(t, o, j_ia, j_hum):
        return guardar_feedback_humano(t if t and t.strip() else o, j_ia, j_hum)
    btn_hitl.click(fn=wrapper_guardar, inputs=[in_txt, out_ocr, out_json, in_hitl], outputs=out_hitl)

# Lanzamiento
app_produccion.launch(debug=True, share=True)

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> None


# **Control de Cambios (Changelog del Proyecto)**
## Control de Cambios y Decisiones Arquitectónicas (V3.0)

**Fecha:** 30-Marzo-2026
**Autor:** Equipo de Investigación (Córdova & Portero)
**Tipo de Actualización:** Refactorización Arquitectónica Core y Optimización de Métricas

### Cambios Implementados:
1. **Migración a Búsqueda Híbrida (Ensemble Retriever):** - *Problema:* El F1-Score se mantenía bajo (~0.33) porque la similitud de cosenos (Vectores Densos) fallaba al buscar SKUs exactos o términos cortos.
   - *Solución:* Se implementó `EnsembleRetriever` combinando `FAISS` (búsqueda semántica) al 50% y `BM25` (búsqueda léxica/keyword) al 50%.
2. **Eliminación de Estrategia de Text Splitting:**
   - *Problema:* El uso de `RecursiveCharacterTextSplitter` rompía el contexto de los registros tabulares.
   - *Solución:* Se configuró una relación 1:1 (Un registro del DataFrame = Un Documento LangChain), preservando la integridad de los atributos técnicos.
3. **Refinamiento de Prompting mediante Chain-of-Thought (CoT):**
   - Se reestructuró el prompt base de `gpt-4o-mini` exigiendo que el modelo genere un `razonamiento` explícito *antes* de asignar la `confianza_ia`, mejorando la precisión en la asignación del tag `REVISION_MANUAL` (True Negatives).
4. **Filtro de Seguridad Post-Inferencia (HITL Override):**
   - Se codificó una capa de seguridad determinista que fuerza la etiqueta `REVISION_MANUAL` si la IA reporta confianza Media/Baja, previniendo alucinaciones en envíos al ERP.

## Control de Cambios y Decisiones Arquitectónicas (V4.0)

**Fecha:** 31-Marzo-2026
**Autor:** Equipo de Investigación (Córdova & Portero)
**Tipo de Actualización:** Corrección de Metodología de Evaluación (Data Drift) y Robustez del Prompt

### Cambios Implementados:
1. **Bootstrapping Dinámico del Ground Truth (Módulo 4):**
   - *Problema:* El F1-Score se estancó en 50% debido a un antipatrón de "Ground Truth Mismatch". Los SKUs en el dataset de prueba estaban "hardcodeados", causando falsos negativos al evaluarlos contra un catálogo que se generaba con IDs aleatorios en cada ejecución.
   - *Solución:* Se implementó la función `generar_dataset_dinamico()`, la cual realiza un muestreo directo de la tabla de dimensiones en memoria (`df_erp`) para construir oraciones de prueba cuyos SKUs tienen garantía matemática de existencia en el espacio latente.
2. **Implementación de Few-Shot Prompting (Módulo 3):**
   - *Mejora:* Se transicionó de un *Zero-Shot Prompt* a un *Few-Shot Prompt*. Se inyectaron pares de ejemplos (Input -> Razonamiento -> JSON) en la plantilla del LLM para blindar la clasificación lógica de entidades fuera de dominio (Out-of-Domain), garantizando el enrutamiento adecuado hacia el Human-In-The-Loop.
3. **Refinamiento de Evaluación Multi-etiqueta (HITL):**
   - *Mejora:* Ajuste en la Teoría de Conjuntos de la matriz de evaluación para que la deducción correcta de incertidumbre (`REVISION_MANUAL`) compute positivamente como un "Verdadero Positivo" en el Recall Global del sistema.

### **Control de Cambios**


*   **29-03-2026**: Se intenta agregar un extra en el codigo en el modulo 4 de modo que se pueda tener una matriz de confusion para evaluar el desempeño y verlo MÓDULO 4.1: Evaluación de Desempeño y Matriz de Confusión Visual integrando las librerías seaborn y matplotlib para generar un gráfico
*   **30-03-2026:** (Cambios)

